# Build the OMNI2 hourly cache for the MagCarto renderings

Writes the OMNI2 cache `omni2_cache.pkl` (at `omni_cache` in `local_paths.json`) in exactly the shape `MagCarto_Master_test.load_omni2_cache()`
expects, so `get_omni2_params()` stops falling back to a live fetch and from there to Wind MFI/SWE
parameters.

**No network access.** The source is `~/.spacepy/data/omni2data.h5`, the OMNI2 database spacepy already
downloaded on this machine: 555 444 hourly records from 1963 to 2026, with all 55 OMNI2 fields under the
same names MagCarto uses. Fill values are already NaN. The earlier version of this notebook downloaded
the yearly `.dat` files from SPDF and timed out behind the university network; that is no longer needed.

Dst comes from OMNI2's own `Dst_index`, which is complete over 1995–2004, so the
`dst{year}.txt` files in the local *Wind* data folder are not used (they are missing 2000, 2002, 2003
and 2004 in any case).

Nothing here touches the figures or the crossing database. Run the cells in order.

## 1. Configuration

In [ ]:
from pathlib import Path
import datetime as dt
import pickle

import h5py                      # pip install h5py, if missing
import numpy as np
import pandas as pd

OMNI2_H5  = Path.home() / ".spacepy" / "data" / "omni2data.h5"
import json

# Machine-specific locations come from local_paths.json beside this notebook. That file is not part of the
# repository: copy local_paths.example.json and fill it in. Without it, BOW_SHOCK is the folder above this one.
_local_file = Path("local_paths.json")
LOCAL = json.loads(_local_file.read_text(encoding="utf-8")) if _local_file.exists() else {}
BOW_SHOCK = Path(LOCAL.get("bow_shock", Path.cwd().parent))
CACHE_OUT = Path(LOCAL.get("omni_cache", BOW_SHOCK / "OMNI" / "omni2_cache.pkl"))   # read by the rendering notebooks

KNOWN_GOOD   = BOW_SHOCK / "Bow_Shock_Stats" / "omni_mach_cache.pkl"   # validation only
CROSSINGS    = BOW_SHOCK / "bs_crossings_V02.xlsx"
COVERAGE_CSV = BOW_SHOCK / "MagCarto" / "omni_coverage_by_leg.csv"
PREVIEW_CSV  = BOW_SHOCK / "MagCarto" / "omni_standoff_preview.csv"

YEAR_FROM, YEAR_TO = 1995, 2005          # [from, to)
MARGIN_H = 3                             # pass-leg window margin, hours

# the five fields a rendering needs: Shue r0 takes Pdyn and Bz, the TS96 parmod takes Pdyn, Dst, By, Bz,
# and the bow shock standoff takes the Alfven Mach number
FIELDS = ["Flow_pressure", "Dst_index", "By_GSM", "Bz_GSM", "Alfven_mach_number"]
EXTRA  = ["Plasma_bulk_speed", "Ion_density", "Plasma_temp", "Plasma_beta", "Field_mag_ave"]

for p in (OMNI2_H5, KNOWN_GOOD, CROSSINGS):
    print(f"{'ok ' if p.exists() else 'MISSING':<8} {p}")
print(f"{'->':<8} {CACHE_OUT}")

## 2. Read the OMNI2 database

`Epoch` is stored as byte strings at hour centres (`1963-01-01T00:30:00`), so the nearest record to any
crossing time is at least 30 minutes away by construction. Nothing downstream cares, but it is why the
offsets reported below are never zero.

In [ ]:
with h5py.File(OMNI2_H5, "r") as f:
    available = set(f.keys())
    missing = [c for c in FIELDS if c not in available]
    if missing:
        raise KeyError(f"omni2data.h5 does not contain {missing}")
    cols = FIELDS + [c for c in EXTRA if c in available]
    df = pd.DataFrame({c: f[c][:].astype(float) for c in cols})
    df["Epoch"] = pd.to_datetime([e.decode() for e in f["Epoch"][:]])

df = df[(df.Epoch >= f"{YEAR_FROM}-01-01") & (df.Epoch < f"{YEAR_TO}-01-01")]
df = df.sort_values("Epoch").reset_index(drop=True)

expected = sum(366 if y % 4 == 0 else 365 for y in range(YEAR_FROM, YEAR_TO)) * 24
print(f"{len(df)} hourly records, {df.Epoch.min()} to {df.Epoch.max()}  (expected {expected})")
print("\nvalid fraction and range per field:")
for c in FIELDS:
    v = df[c]
    print(f"   {c:<20} {100*v.notna().mean():5.1f} %   "
          f"min {np.nanmin(v):9.2f}  max {np.nanmax(v):9.2f}  NaN {v.isna().sum():6d}")

## 3. Validate

Physical ranges, then a comparison against `omni_mach_cache.pkl`, which was built independently. If the
two disagree, something is wrong with the source file and the cache should not be written.

In [ ]:
RANGES = {"Flow_pressure": (0.0, 80.0), "Dst_index": (-600.0, 100.0),
          "By_GSM": (-60.0, 60.0), "Bz_GSM": (-60.0, 60.0),
          "Alfven_mach_number": (0.5, 200.0)}
problems = []
for c, (lo, hi) in RANGES.items():
    v = df[c].dropna()
    out = int(((v < lo) | (v > hi)).sum())
    print(f"   {c:<20} outside {lo}..{hi}: {out}")
    if out:
        problems.append(f"{c}: {out} values out of range")

ref = pickle.load(open(KNOWN_GOOD, "rb")).copy()
ref["time"] = pd.to_datetime(ref["time"])
# omni_mach_cache is on the hour, omni2data.h5 on the half hour: match on the containing hour
a = df.assign(hour=df.Epoch.dt.floor("h"))
m = a.merge(ref.assign(hour=ref.time.dt.floor("h")), on="hour", how="inner")
print(f"\ncross-check against {KNOWN_GOOD.name}: {len(m)} hours in common")
for mine, theirs, tol in (("Alfven_mach_number", "alfven_mach", 0.15),
                          ("Flow_pressure", "flow_pressure_npa", 0.02),
                          ("Plasma_bulk_speed", "vsw_km_s", 1.0)):
    if mine not in m or theirs not in m:
        continue
    both = m[[mine, theirs]].dropna()
    diff = (both[mine] - both[theirs]).abs()
    frac = float((diff <= tol).mean()) if len(both) else 0.0
    print(f"   {mine:<20} vs {theirs:<20} n={len(both):6d}  median diff {diff.median():7.3f}  "
          f"within {tol}: {100*frac:5.1f} %")
    if len(both) < 1000 or frac < 0.98:
        problems.append(f"{mine} disagrees with {theirs}")

if problems:
    for p in problems:
        print("\n!!", p)
    raise SystemExit("validation failed; do not use this cache")
print("\nvalidation passed")

## 4. Coverage over the 140 pass legs

A leg can be re-rendered only if some hour in its window has **all five** fields valid at once. In
OMNI2 the IMF and the plasma records have different gaps: `Dst_index` is complete over this interval and
`Bz_GSM` is 99.5 % complete, so the binding fields are `Flow_pressure` and `Alfven_mach_number`.

In [ ]:
x = pd.read_excel(CROSSINGS, sheet_name=0)
x["time"] = pd.to_datetime(x["time"])
x["start time"] = pd.to_datetime(x["start time"])
# A leg is a run of consecutive rows with the same pass and SC direction. Until October 2026 the window
# columns were filled only on the first row of each leg and legs were found from the blank cells; the
# windows are now repeated on every row, so the leg is taken from pass and direction instead. On the
# old file both rules give the same 140 legs.
_key = x[["pass", "SC direction"]]
x["leg"] = (_key != _key.shift()).any(axis=1).cumsum()
assert x.groupby(["pass", "SC direction"])["leg"].nunique().max() == 1, \
    "a (pass, SC direction) pair is split into more than one block of rows"

o = df.set_index("Epoch")
rows = []
for leg, sub in x.groupby("leg"):
    t0 = sub["time"].min() - pd.Timedelta(hours=MARGIN_H)
    t1 = sub["time"].max() + pd.Timedelta(hours=MARGIN_H)
    w = o[(o.index >= t0) & (o.index <= t1)]
    rows.append(dict(leg=int(leg), pass_id=int(sub["pass"].iloc[0]),
                     sc_direction=sub["SC direction"].iloc[0],
                     first_crossing=sub["time"].min(), n_crossings=len(sub), window_hours=len(w),
                     **{f"n_{c}": int(w[c].notna().sum()) for c in FIELDS},
                     n_complete=int(w[FIELDS].notna().all(axis=1).sum())))
cov = pd.DataFrame(rows)

print(f"{len(cov)} pass legs, window = first crossing -{MARGIN_H} h to last +{MARGIN_H} h\n")
for c in FIELDS:
    print(f"   {c:<20} legs with no valid hour: {(cov['n_'+c]==0).sum():3d}")
print(f"   {'all five together':<20} legs with no valid hour: {(cov.n_complete==0).sum():3d}")

skip = cov[cov.n_complete == 0]
print(f"\nSKIP LIST -- cannot be re-rendered from OMNI ({len(skip)} legs):")
print(skip[["pass_id", "sc_direction", "first_crossing", "window_hours"]
           + [f"n_{c}" for c in FIELDS]].to_string(index=False))

thin = cov[(cov.n_complete > 0) & (cov.n_complete < 0.5 * cov.window_hours)]
print(f"\nunder 50 % complete coverage but usable ({len(thin)} legs):")
print(thin[["pass_id", "sc_direction", "first_crossing", "window_hours", "n_complete"]]
      .to_string(index=False))

cov.to_csv(COVERAGE_CSV, index=False)
print(f"\nwrote {COVERAGE_CSV}")

## 5. Standoff preview — does the 1–10 R_E clamp ever bind?

Shue r0 and the bow shock standoff for every leg, from the nearest hour where all five fields are valid.
This is what decides whether `delta_min` / `delta_max` in `bowshock_rtheta_from_Ma` can simply be
dropped.

In [ ]:
GAMMA = 5.0 / 3.0

def shue_r0(pdyn_npa, bz_nt):
    """Shue et al. (1998) Eq. 10."""
    return (10.22 + 1.29 * np.tanh(0.184 * (bz_nt + 8.14))) * pdyn_npa ** (-1.0 / 6.6)

def standoff(r0_mp, mach):
    """Gas-dynamic standoff scaling, unclamped. Diverges as mach -> 1."""
    return r0_mp * ((GAMMA - 1.0) * mach**2 + 2.0) / ((GAMMA + 1.0) * (mach**2 - 1.0))

complete = df.dropna(subset=FIELDS).reset_index(drop=True)
res = []
for _, r in cov.iterrows():
    k = complete.Epoch.searchsorted(r.first_crossing)
    cand = [i for i in (k - 1, k) if 0 <= i < len(complete)]
    i = min(cand, key=lambda i: abs((complete.Epoch[i] - r.first_crossing).total_seconds()))
    row = complete.loc[i]
    r0 = shue_r0(row.Flow_pressure, row.Bz_GSM)
    ma = row.Alfven_mach_number
    res.append(dict(pass_id=r.pass_id, sc_direction=r.sc_direction,
                    skipped=bool(r.n_complete == 0),
                    offset_h=(row.Epoch - r.first_crossing).total_seconds() / 3600,
                    Pdyn=row.Flow_pressure, Bz=row.Bz_GSM, By=row.By_GSM, Dst=row.Dst_index,
                    Ma=ma, r0=r0, delta=standoff(r0, ma) if ma > 1.0 else np.nan))
pre = pd.DataFrame(res)
usable = pre[~pre.skipped]

print(f"legs with a complete hour within 1 h: {(usable.offset_h.abs()<=1).sum()} of {len(usable)}"
      f"   3 h: {(usable.offset_h.abs()<=3).sum()}   6 h: {(usable.offset_h.abs()<=6).sum()}")
print(f"\nAlfven Mach number: min {usable.Ma.min():.2f}  10th {usable.Ma.quantile(.1):.2f}  "
      f"median {usable.Ma.median():.1f}  max {usable.Ma.max():.1f}")
print(f"Shue r0           : min {usable.r0.min():.1f}  median {usable.r0.median():.1f}  "
      f"max {usable.r0.max():.1f} R_E")
print(f"standoff delta    : min {usable.delta.min():.2f}  median {usable.delta.median():.2f}  "
      f"max {usable.delta.max():.2f} R_E")
print(f"\nlegs where delta would exceed 10 R_E : {(usable.delta>10).sum()}")
print(f"legs where delta would fall below 1 R_E: {(usable.delta<1).sum()}")
print(f"legs with Ma <= 1.2 (relation runs away): {(usable.Ma<=1.2).sum()}")

pre.to_csv(PREVIEW_CSV, index=False)
print(f"\nwrote {PREVIEW_CSV}")

## 6. Write the cache

In [ ]:
cache = {"Epoch": list(df.Epoch.dt.to_pydatetime())}
for c in FIELDS + [c for c in EXTRA if c in df.columns]:
    cache[c] = df[c].to_numpy(dtype=float)
cache["_built"] = dt.datetime.now().isoformat(timespec="seconds")
cache["_source"] = str(OMNI2_H5)
cache["_interval"] = f"{YEAR_FROM}-{YEAR_TO-1}"

CACHE_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(CACHE_OUT, "wb") as fh:
    pickle.dump(cache, fh)
print(f"wrote {CACHE_OUT}  ({CACHE_OUT.stat().st_size/1e6:.1f} MB)")
print("keys:", [k for k in cache if not k.startswith('_')])

## 7. Verify it the way MagCarto will read it

Reproduces the lookup in `get_omni2_params()`. If these print sensible numbers, the renderer will work
off the cache instead of attempting a live fetch.

In [ ]:
loaded = pickle.load(open(CACHE_OUT, "rb"))
cache_times = list(loaded["Epoch"])

def get_omni2_params_like_magcarto(t_req):
    idx = int(np.argmin([abs((c - t_req).total_seconds()) for c in cache_times]))
    return (float(loaded["Flow_pressure"][idx]), float(loaded["Dst_index"][idx]),
            float(loaded["By_GSM"][idx]), float(loaded["Bz_GSM"][idx]),
            float(loaded["Alfven_mach_number"][idx]), cache_times[idx])

for t in [dt.datetime(1995, 8, 1, 21, 37), dt.datetime(1996, 5, 9, 14, 0),
          dt.datetime(2000, 5, 2, 7, 45), dt.datetime(2004, 4, 27, 4, 48)]:
    Pdyn, Dst, By, Bz, Ma, used = get_omni2_params_like_magcarto(t)
    print(f"{t}  ->  {used}   Pdyn={Pdyn:6.2f}  Dst={Dst:7.1f}  By={By:+6.1f}  "
          f"Bz={Bz:+6.1f}  Ma={Ma:6.1f}")

print("\nThat lookup is a linear scan over 87 672 epochs per call, which is how MagCarto does it:")
print("a second or two per pass. Swapping it for np.searchsorted on the Epoch array is a one-line")
print("change in get_omni2_params() if the 140-pass loop feels slow.")

## Done

Expected results, already computed from the same source file:

- **4 legs cannot be re-rendered**: passes 4, 22, 55 and 56 inbound. In every case the missing fields are
  `Flow_pressure` and `Alfven_mach_number`; `Dst_index` is present for all four and `By`/`Bz` for three
  of them.
- 128 of the remaining 136 legs have a complete hour within 1 h, 135 within 3 h, 138 within 6 h.
- Alfven Mach number over the usable legs: min **2.50**, median 8.0, max 39.7.
- Standoff distance: min 1.91, median 2.72, **max 4.99 R_E**.
- Legs where the clamp would bind: **0**. Legs with Ma <= 1.2: **0**.

So `delta_min` / `delta_max` can be deleted outright — with real upstream parameters the standoff never
approaches the old 10 R_E cap. If cells 4 and 5 reproduce these numbers, the cache is good and the
rendering notebook can be written against the four-leg skip list.